In [ ]:
!pip install spacy

In [ ]:
!python -m spacy download en_core_web_md

In [ ]:
import pandas as pd
import os
import spacy
import re
import time

In [ ]:
# Nombre de la subcarpeta para este experimento
EXPERIMENT_FOLDER = "Exp01_Solo_Lema"

# Interruptores de Ablación (Solo Lematización activada)
USE_LEMMATIZATION = True
DROP_STOPWORDS = False
DROP_PUNCTUATION = False
NORMALIZE_ELONGATION = False

# Rutas S3
INPUT_TRAIN_PREFIX = ".\limpieza_minima/train_parquet"
OUTPUT_TRAIN_PREFIX = f"Spacy/{EXPERIMENT_FOLDER}/train_parquet"

TEXT_COLUMN = "clean_text"

In [ ]:
print("Cargando modelo de spaCy (Optimizado)...")

nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

In [ ]:
def ablation_clean(text: str) -> str:
    if pd.isna(text):
        return ""
        
    text = str(text).lower()
    
    if NORMALIZE_ELONGATION:
        text = re.sub(r'(.)\1{2,}', r'\1', text)
        
    doc = nlp(text)
    tokens = []
    
    for token in doc:
        if DROP_STOPWORDS and token.is_stop: continue
        if DROP_PUNCTUATION and token.is_punct: continue
            
        word = token.lemma_ if USE_LEMMATIZATION else token.text
        tokens.append(word)
        
    return " ".join(tokens)

In [ ]:
def run_etl():
    # Crear carpeta de salida si no existe
    os.makedirs(OUTPUT_TRAIN_PREFIX, exist_ok=True)

    files = [
        os.path.join(INPUT_TRAIN_PREFIX, f)
        for f in os.listdir(INPUT_TRAIN_PREFIX)
        if f.endswith(".parquet")
    ]

    if not files:
        print(f"No se encontraron archivos en {INPUT_TRAIN_PREFIX}")
        return

    print(f"Iniciando procesamiento para: {EXPERIMENT_FOLDER}")
    start_time = time.time()

    for file in files:
        filename = os.path.basename(file)
        output_path = os.path.join(OUTPUT_TRAIN_PREFIX, filename)

        # Validar si ya se procesó
        if os.path.exists(output_path):
            print(f"  ⏭️ Partición {filename} ya existe. Saltando...")
            continue

        print(f"Procesando: {filename}...")
        df = pd.read_parquet(file)

        # Aplicamos la limpieza
        df["ablation_text"] = df[TEXT_COLUMN].apply(ablation_clean)

        # Guardamos en carpeta local
        df.to_parquet(
            output_path,
            engine="pyarrow",
            index=False,
            compression="snappy"
        )

        print(f" Guardado en {output_path}")

    elapsed = time.time() - start_time
    print(f"Experimento '{EXPERIMENT_FOLDER}' completado en {elapsed/60:.2f} minutos.")


# ¡Ejecutar!
run_etl()